# 02 — Cleaning (Missing values & outliers)

## Objectif

- Produire une version "clean" du dataset prête pour la création de features et la modélisation, en appliquant des règles de nettoyage reproductibles et sans fuite.


## 1. Chargement des données parsées (interim)

In [16]:
import numpy as np
import pandas as pd
from pathlib import Path
import json

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path("..").resolve()

# Data
IN_PATH = PROJECT_ROOT / "data" / "interim" / "base1_parsed.parquet"
OUT_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Artifacts Notebook 02
ART_02 = PROJECT_ROOT / "artifacts" / "02_cleaning"
ART_02_SUM = ART_02 / "summary"
ART_02_TBL = ART_02 / "tables"
ART_02_PLOT = ART_02 / "plots"

for d in [ART_02, ART_02_SUM, ART_02_TBL, ART_02_PLOT]:
    d.mkdir(parents=True, exist_ok=True)

assert IN_PATH.exists(), f"Fichier manquant: {IN_PATH}"

df = pd.read_parquet(IN_PATH)
print("shape:", df.shape)
print("stations:", df["id_sonde"].nunique())
print("Min ts:", df["ts"].min(), "| Max ts:", df["ts"].max())

display(df.head(3))

shape: (164206, 15)
stations: 7
Min ts: 2013-05-29 14:00:00 | Max ts: 2018-10-05 08:00:00


,row_id,id_sonde,ts,date,ts_raw,date_raw,temp_water_c,temp_air_eobs_c,rainf_eobs,discharge_q,discharge_q_qjm,discharge_q_mmj,rain_rr_ref_eobs,rain_rr_mean_eobs,temp_tg_ref_eobs
0,46917,817,2013-05-29 14:00:00,2013-05-29,2013-05-29 14:00:00,2013-05-29,12.703,9.75,2.8,18.1,18.1,0.717358,2.8,3.360976,9.75
1,46918,817,2013-05-29 16:00:00,2013-05-29,2013-05-29 16:00:00,2013-05-29,12.896,9.75,2.8,18.1,18.1,0.717358,2.8,3.360976,9.75
2,46919,817,2013-05-29 18:00:00,2013-05-29,2013-05-29 18:00:00,2013-05-29,12.968,9.75,2.8,18.1,18.1,0.717358,2.8,3.360976,9.75


## 2. Diagnostic initial (colonnes & valeurs manquantes)

- On identifie la variable cible, les colonnes temps/identifiants, puis on liste les variables exogènes.  
- On calcule ensuite le taux de valeurs manquantes des exogènes avant nettoyage.


In [10]:
target = "temp_water_c"
time_cols = ["id_sonde", "ts", "date"]
raw_cols = ["ts_raw", "date_raw", "row_id"]

# exogznes = toutes sauf identifiants/temps/target/raw
exclude = set(time_cols + raw_cols + [target])
exo_cols = [c for c in df.columns if c not in exclude]

print("Exogènes:", exo_cols)

missing_before = (df[exo_cols].isna().mean() * 100).sort_values(ascending=False).to_frame("pct_missing")
missing_before["n_missing"] = df[exo_cols].isna().sum()
display(missing_before)

Exogènes: ['temp_air_eobs_c', 'rainf_eobs', 'discharge_q', 'discharge_q_qjm', 'discharge_q_mmj', 'rain_rr_ref_eobs', 'rain_rr_mean_eobs', 'temp_tg_ref_eobs']


,pct_missing,n_missing
discharge_q,0.619953,1018
discharge_q_mmj,0.619953,1018
rain_rr_ref_eobs,0.495719,814
rain_rr_mean_eobs,0.495719,814
temp_tg_ref_eobs,0.495719,814
discharge_q_qjm,0.336163,552
temp_air_eobs_c,0.000000,0
rainf_eobs,0.000000,0


## 3. Stratégie de nettoyage (règles)

**Variables concernées.** Nettoyage appliqué uniquement aux **variables exogènes** (la cible `temp_water_c` n’a pas de valeurs manquantes).

**Valeurs manquantes.**
- Variables “lisses” (débit, `temp_tg_ref_eobs`) : **forward-fill causal** par station, avec une limite (ex. ≤ 12h).
- Variables de pluie (`rain_rr_*`) : remplacement des `NaN` par **0** (hypothèse “pas de précipitation mesurée”), avec conservation d’un **flag**.

**Outliers dynamiques.** Les spikes (variations trop rapides) sont **documentés/flag** et seront traités si nécessaire, de manière causale.

**Traçabilité.** Pour chaque variable imputée, on conserve une colonne `__was_missing` et on produit un rapport avant/après.


In [11]:

FFILL_LIMIT_STEPS = 6  # 6 pas de 2h = 12h 

df_clean = df.sort_values(["id_sonde", "ts"]).copy()

n_dup0 = int(df_clean.duplicated(["id_sonde", "ts"]).sum())
if n_dup0 > 0:
    df_clean = df_clean.drop_duplicates(["id_sonde", "ts"], keep="first").copy()
    print(f"doublon supprimés avant cleaning : {n_dup0}")

slow_cols = [c for c in ["discharge_q", "discharge_q_qjm", "discharge_q_mmj", "temp_tg_ref_eobs"] if c in exo_cols]
rain_cols = [c for c in ["rain_rr_ref_eobs", "rain_rr_mean_eobs"] if c in exo_cols]

print("slow_cols:", slow_cols)
print("rain_cols:", rain_cols)

# Flags de traçabilité 
for c in slow_cols + rain_cols:
    df_clean[f"{c}__was_missing"] = df_clean[c].isna().astype("int8")

# var lisses : forward-fill limité par station
if slow_cols:
    df_clean[slow_cols] = df_clean.groupby("id_sonde")[slow_cols].ffill(limit=FFILL_LIMIT_STEPS)

# Var pluie : remplacer NaN par 0 + flag conservé
if rain_cols:
    df_clean[rain_cols] = df_clean[rain_cols].fillna(0.0)

missing_after = (df_clean[exo_cols].isna().sum()).to_frame("n_missing_after")
missing_after["pct_missing_after"] = (df_clean[exo_cols].isna().mean() * 100)

# Comparatif avant/après 
compare = missing_before.copy()
compare = compare.rename(columns={"n_missing": "n_missing_before", "pct_missing": "pct_missing_before"})
compare = compare.join(missing_after)

compare["n_imputed"] = compare["n_missing_before"] - compare["n_missing_after"]
compare = compare.sort_values("n_missing_before", ascending=False)

display(compare)

print("\nColonnes exogènes encore manquantes après règle :", compare[compare["n_missing_after"] > 0].index.tolist())


slow_cols: ['discharge_q', 'discharge_q_qjm', 'discharge_q_mmj', 'temp_tg_ref_eobs']
rain_cols: ['rain_rr_ref_eobs', 'rain_rr_mean_eobs']


,pct_missing_before,n_missing_before,n_missing_after,pct_missing_after,n_imputed
discharge_q,0.619953,1018,946,0.576106,72
discharge_q_mmj,0.619953,1018,946,0.576106,72
rain_rr_ref_eobs,0.495719,814,0,0.000000,814
rain_rr_mean_eobs,0.495719,814,0,0.000000,814
temp_tg_ref_eobs,0.495719,814,754,0.459179,60
discharge_q_qjm,0.336163,552,498,0.303278,54
temp_air_eobs_c,0.000000,0,0,0.000000,0
rainf_eobs,0.000000,0,0,0.000000,0



Colonnes exogènes encore manquantes après règle : ['discharge_q', 'discharge_q_mmj', 'temp_tg_ref_eobs', 'discharge_q_qjm']


### Bilan du traitement des valeurs manquantes (exogènes)

- Les variables de pluie `rain_rr_*` ont été imputées à 0 (absence de mesure assimilée à 0), avec conservation d’un indicateur `__was_missing` pour la traçabilité.
- Les variables “lisses” (`discharge_*`, `temp_tg_ref_eobs`) ont été imputées par **forward-fill causal** par station avec une limite de 12h (6 pas de 2h).  
  Cette règle comble uniquement les trous courts ; des valeurs manquantes peuvent subsister pour des segments plus longs ou en début de série.
- Un rapport avant/après est produit afin de quantifier les valeurs imputées et les valeurs manquantes restantes.

## 4. Détection et traçabilité des spikes (cible)

**Objectif.** Identifier les variations anormalement rapides de `temp_water_c` (spikes) afin de les documenter et, si besoin, de les traiter plus tard sans altérer la reproductibilité.

**Principe.** Calcul de `dTeau` sur un pas de 2 heures par station et création d’un indicateur `is_spike_temp_water` (seuil initial conservateur, ex. 3°C/2h).

**Remarque.** Dans ce notebook, on **flag** les spikes (traçabilité) sans modifier la cible ; la correction éventuelle sera discutée/justifiée séparément.

In [12]:

THRESH_SPIKE = 3.0  

df_clean = df_clean.sort_values(["id_sonde", "ts"]).copy()
df_clean["dtemp_2h"] = df_clean.groupby("id_sonde")["temp_water_c"].diff()
df_clean["is_spike_temp_water"] = (df_clean["dtemp_2h"].abs() > THRESH_SPIKE).astype("int8")

n_spikes = int(df_clean["is_spike_temp_water"].sum())
print(f"Nb spikes |dTeau| > {THRESH_SPIKE} °C/2h :", n_spikes)

if n_spikes > 0:
    display(df_clean[df_clean["is_spike_temp_water"] == 1][
        ["id_sonde", "ts", "temp_water_c", "dtemp_2h"]])

Nb spikes |dTeau| > 3.0 °C/2h : 1


,id_sonde,ts,temp_water_c,dtemp_2h
5537,817,2014-09-03,13.882,-3.367


## 5. Contrôles post-nettoyage & export (processed)

**Objectif.** Vérifier que le dataset nettoyé respecte les invariants (tri temporel, absence de doublons, timestamps valides) et exporter une version `processed` prête pour **définir le split temporel** et lancer le feature engineering.

**Sorties attendues.**
- Un résumé des valeurs manquantes restantes **sur les exogènes**.
- Un fichier `base1_clean.parquet` dans `data/processed/`.
- Un petit rapport `cleaning_report.csv` (traçabilité).

In [13]:
STATION_COL = "id_sonde"
TS_COL = "ts"

assert df_clean.duplicated([STATION_COL, TS_COL]).sum() == 0, "Doublons détectés après cleaning"
assert df_clean[TS_COL].isna().sum() == 0, "NaT détectés dans ts"
print("verif OK")

missing_after_simple = (
    (df_clean[exo_cols].isna().mean() * 100)
    .sort_values(ascending=False)
    .to_frame("pct_missing_after"))
missing_after_simple["n_missing_after"] = df_clean[exo_cols].isna().sum()
display(missing_after_simple)

#Export processed 
out_path = OUT_DIR / "base1_clean.parquet"
df_clean.to_parquet(out_path, index=False)
print("Saved:", out_path)

# Report global (CSV traçabilité)
spike_col = "is_spike_temp_water"
report = pd.DataFrame([{
    "n_rows": int(len(df_clean)),
    "n_stations": int(df_clean[STATION_COL].nunique()),
    "min_ts": str(df_clean[TS_COL].min()),
    "max_ts": str(df_clean[TS_COL].max()),
    "n_spikes_flagged": int(df_clean[spike_col].sum()) if spike_col in df_clean.columns else None,
    "exo_missing_total_after": int(df_clean[exo_cols].isna().sum().sum()),}])
display(report)

report_path = ART_02_SUM / "cleaning_report.csv"
report.to_csv(report_path, index=False)
print("Saved:", report_path)

verif OK


,pct_missing_after,n_missing_after
discharge_q,0.576106,946
discharge_q_mmj,0.576106,946
temp_tg_ref_eobs,0.459179,754
discharge_q_qjm,0.303278,498
temp_air_eobs_c,0.000000,0
rainf_eobs,0.000000,0
rain_rr_ref_eobs,0.000000,0
rain_rr_mean_eobs,0.000000,0


Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\data\processed\base1_clean.parquet


,n_rows,n_stations,min_ts,max_ts,n_spikes_flagged,exo_missing_total_after
0,164206,7,2013-05-29 14:00:00,2018-10-05 08:00:00,1,3144


Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\02_cleaning\summary\cleaning_report.csv


In [14]:
# longueur max de trous NA consécutif
def max_na_run(s: pd.Series) -> int:
    if s.isna().sum() == 0:
        return 0
    return int(s.isna().astype(int).groupby(s.notna().cumsum()).sum().max())

rows = []
for sid, gg in df_clean.sort_values([STATION_COL, TS_COL]).groupby(STATION_COL):
    for c in exo_cols:
        run_steps = max_na_run(gg[c])
        rows.append({
            STATION_COL: sid,
            "var": c,
            "max_na_run_steps": run_steps,
            "max_na_run_hours": run_steps * 2,})

na_runs = pd.DataFrame(rows).sort_values("max_na_run_steps", ascending=False)
display(na_runs.head(10))

max_gap_by_var = (na_runs.groupby("var")["max_na_run_hours"].max()
                  .sort_values(ascending=False).to_frame("max_gap_hours"))
print("Top variable avec plus gros trous (heures):")
display(max_gap_by_var)

,id_sonde,var,max_na_run_steps,max_na_run_hours
28,825,discharge_q_mmj,258,516
26,825,discharge_q,258,516
31,825,temp_tg_ref_eobs,234,468
47,828,temp_tg_ref_eobs,95,190
36,827,discharge_q_mmj,95,190
39,827,temp_tg_ref_eobs,95,190
42,828,discharge_q,95,190
44,828,discharge_q_mmj,95,190
34,827,discharge_q,95,190
43,828,discharge_q_qjm,90,180


Top variable avec plus gros trous (heures):


,max_gap_hours
var,
discharge_q,516
discharge_q_mmj,516
temp_tg_ref_eobs,468
discharge_q_qjm,180
rain_rr_mean_eobs,0
rain_rr_ref_eobs,0
rainf_eobs,0
temp_air_eobs_c,0


In [15]:
#json résumé (cleaning)
summary = report.iloc[0].to_dict()
summary.update({
    "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    "expected_freq": "2H",
    "stage": "cleaning",})
summary_path = ART_02_SUM / "QC_cleaning_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print("Saved:", summary_path)

# Tables (CSV)
missing_after_simple.reset_index().rename(columns={"index": "var"}).to_csv(
    ART_02_TBL / "exo_missing_after.csv", index=False)
na_runs.to_csv(ART_02_TBL / "exo_na_max_run_by_station_var.csv", index=False)
max_gap_by_var.reset_index().to_csv(ART_02_TBL / "exo_na_max_gap_by_var.csv", index=False)
compare.reset_index().rename(columns={"index": "var"}).to_csv(
    ART_02_TBL / "exo_missing_before_after.csv", index=False)

print("Saved:", ART_02_TBL / "exo_missing_after.csv")
print("Saved:", ART_02_TBL / "exo_na_max_run_by_station_var.csv")
print("Saved:", ART_02_TBL / "exo_na_max_gap_by_var.csv")

Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\02_cleaning\summary\QC_cleaning_summary.json
Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\02_cleaning\tables\exo_missing_after.csv
Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\02_cleaning\tables\exo_na_max_run_by_station_var.csv
Saved: C:\Users\El-fahad COMBO\Desktop\Serie_temporelle\artifacts\02_cleaning\tables\exo_na_max_gap_by_var.csv


### Synthèse nettoyage (Notebook 02)

- Un unique spike a été détecté et **flaggé** sur la cible `temp_water_c` (station 817, 2014-09-03 00:00, Δ=-3.367°C). La cible n’a pas été modifiée à ce stade ; l’indicateur `is_spike_temp_water` assure la traçabilité.
- Les valeurs manquantes des exogènes ont été traitées de manière **causale** et **par station** :
  - `rain_rr_*` : `NaN` remplacés par 0 + indicateurs `__was_missing`.
  - Variables “lisses” (`discharge_*`, `temp_tg_ref_eobs`) : **forward-fill limité à 6 pas (12h)**, ce qui laisse subsister des manquants sur les segments plus longs ou en début de série.
- Après nettoyage, il reste un faible volume de manquants exogènes (3144 valeurs au total). Les diagnostics montrent que la proportion globale est faible, mais que certaines stations présentent des **trous consécutifs longs** (jusqu’à plusieurs jours), ce qui justifie de ne pas imputer agressivement.
- Les données nettoyées ont été exportées dans `data/processed/base1_clean.parquet` ainsi qu’un rapport `cleaning_report.csv`. Les tables de diagnostic (manquants et longueurs de trous) sont sauvegardées dans `artifacts/tables/`, et le résumé dans `artifacts/summary/QC_cleaning_summary.json`.